# 04 — Random Walk
**Week 2 | Mathematical Foundations for RL**

Random walks appear everywhere in RL — from the way an agent explores to the theoretical analysis
of TD learning. They also give great intuition about variance in stochastic processes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(7)

## 1. Simple 1D Random Walk
At each step: move +1 (right) or -1 (left) with equal probability.

In [ ]:
def random_walk_1d(n_steps, p_right=0.5):
    steps = np.where(np.random.rand(n_steps) < p_right, 1, -1)
    return np.concatenate([[0], np.cumsum(steps)])

n_steps = 500
fig, ax = plt.subplots(figsize=(10, 3.5))
for i in range(20):
    walk = random_walk_1d(n_steps)
    ax.plot(walk, alpha=0.4, linewidth=0.8)
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Step'); ax.set_ylabel('Position')
ax.set_title('20 Random Walks (1D, 500 steps)')
plt.tight_layout(); plt.show()

## 2. Distribution of Positions at Time t
At time t, position X_t ~ N(0, t) — variance grows linearly with time.

In [ ]:
checkpoints = [10, 50, 100, 500]
n_walks = 10_000
fig, axes = plt.subplots(1, len(checkpoints), figsize=(14, 3), sharey=False)

for ax, t in zip(axes, checkpoints):
    positions = [random_walk_1d(t)[-1] for _ in range(n_walks)]
    ax.hist(positions, bins=40, color='steelblue', edgecolor='white', density=True)
    ax.set_title(f't = {t}\nstd ≈ {np.std(positions):.1f}')
    ax.set_xlabel('Position')

axes[0].set_ylabel('Density')
plt.suptitle('Distribution of position at time t', y=1.02)
plt.tight_layout(); plt.show()
print("Theoretical std = sqrt(t):", [f"{t}→{t**0.5:.1f}" for t in checkpoints])

## 3. Biased Random Walk
What if the agent has a preference? (p_right > 0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for p, color, label in [(0.5,'gray','p=0.5 (unbiased)'), (0.55,'steelblue','p=0.55'), (0.6,'seagreen','p=0.60')]:
    walks = np.array([random_walk_1d(500, p) for _ in range(500)])
    mean_walk = walks.mean(axis=0)
    std_walk  = walks.std(axis=0)
    x = np.arange(501)
    ax.plot(mean_walk, color=color, linewidth=2, label=label)
    ax.fill_between(x, mean_walk-std_walk, mean_walk+std_walk, alpha=0.15, color=color)
ax.set_xlabel('Step'); ax.set_ylabel('Mean position ± std')
ax.set_title('Biased vs Unbiased Random Walk')
ax.legend(); plt.tight_layout(); plt.show()

## 4. First Passage Time
How long until the walk reaches position +10 for the first time?

In [ ]:
def first_passage_time(target=10, max_steps=5000):
    pos = 0
    for t in range(1, max_steps+1):
        pos += np.random.choice([-1, 1])
        if pos >= target:
            return t
    return max_steps  # didn't reach target

fpt = [first_passage_time(10) for _ in range(5000)]
plt.figure(figsize=(7, 3))
plt.hist(fpt, bins=60, color='darkorange', edgecolor='white')
plt.xlabel('Steps to reach position +10'); plt.ylabel('Count')
plt.title(f'First Passage Time Distribution (mean={np.mean(fpt):.0f})')
plt.tight_layout(); plt.show()

## ✅ Exercises
1. Modify the 1D walk to stop when it hits +20 or -20. What fraction of walks end at +20 vs -20?
2. Simulate a **2D random walk** (move up/down/left/right). Plot 5 trajectories on an x-y grid.
3. **Challenge**: implement the classic '5-state random walk' from Sutton & Barto Example 6.2. States A–E, terminal states at each end. Compute true state values analytically and verify empirically.

## Q1
When the walk is unbiased (p = 0.5), approximately 50% of walks end at +20 and 50% end at -20, since there is no directional preference.

In [ ]:
def random_walk_bounded(bound=20, max_steps=100000):
    pos = 0
    for _ in range(max_steps):
        pos += np.random.choice([-1, 1])
        if pos >= bound or pos <= -bound:
            return pos
    return pos

n_trials = 10000
endpoints = [random_walk_bounded(20) for _ in range(n_trials)]

frac_pos = sum(e == 20 for e in endpoints) / n_trials
frac_neg = sum(e == -20 for e in endpoints) / n_trials
print(f"Fraction ending at +20: {frac_pos:.3f}")
print(f"Fraction ending at -20: {frac_neg:.3f}")

## Q2

In [ ]:
def random_walk_2d(n_steps):
    directions = np.array([[0,1],[0,-1],[1,0],[-1,0]])
    moves = directions[np.random.randint(0, 4, n_steps)]
    path = np.concatenate([[[0,0]], np.cumsum(moves, axis=0)])
    return path

fig, ax = plt.subplots(figsize=(6, 6))
colors = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'purple']
for i in range(5):
    path = random_walk_2d(1000)
    ax.plot(path[:, 0], path[:, 1], alpha=0.7, linewidth=0.8, color=colors[i], label=f'Walk {i+1}')
    ax.plot(0, 0, 'ko', markersize=5)
    ax.plot(path[-1, 0], path[-1, 1], 'x', color=colors[i], markersize=8)

ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.set_title('2D Random Walk — 5 Trajectories')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## Q3
States: A, B, C, D, E with terminal states at each end (Left = 0, Right = 1 reward). True values are V(A)=1/6, V(B)=2/6, V(C)=3/6, V(D)=4/6, V(E)=5/6.

In [ ]:
# States: 0=Left Terminal, 1=A, 2=B, 3=C, 4=D, 5=E, 6=Right Terminal
true_values = np.array([0, 1/6, 2/6, 3/6, 4/6, 5/6, 1])
state_names  = ['L', 'A', 'B', 'C', 'D', 'E', 'R']

def run_episode():
    """Returns list of (state, reward) pairs for one episode."""
    state = 3  # start at C
    trajectory = [state]
    while state not in [0, 6]:
        state += np.random.choice([-1, 1])
        trajectory.append(state)
    return trajectory

# TD(0) to estimate values
def td_learning(n_episodes=10000, alpha=0.1, gamma=1.0):
    V = np.full(7, 0.5)
    V[0] = 0; V[6] = 1  # terminal states fixed
    for _ in range(n_episodes):
        traj = run_episode()
        for i in range(len(traj) - 1):
            s  = traj[i]
            s_ = traj[i+1]
            r  = 1 if s_ == 6 else 0
            V[s] += alpha * (r + gamma * V[s_] - V[s])
        V[0] = 0; V[6] = 1
    return V

V_empirical = td_learning(n_episodes=10000)

# Plot analytical vs empirical
plt.figure(figsize=(7, 4))
plt.plot(state_names, true_values, 'o--', color='tomato', label='True values')
plt.plot(state_names, V_empirical, 's-', color='steelblue', label='TD(0) estimate')
plt.xlabel('State'); plt.ylabel('Value')
plt.title('5-State Random Walk — True vs TD(0) Estimated Values')
plt.legend(); plt.tight_layout(); plt.show()

print("True values:      ", np.round(true_values, 4))
print("TD(0) estimates:  ", np.round(V_empirical, 4))

The TD(0) estimates converge closely to the analytically derived true values V(A)=0.167 through V(E)=0.833, confirming the empirical result matches theory.